In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/campaign_table.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/causal_data.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/coupon.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/campaign_desc.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/product.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/transaction_data.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/hh_demographic.csv
/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/coupon_redempt.csv


In [2]:

import os
import pandas as pd

base_path = (
    "/kaggle/input/datasets/frtgnn/"
    "dunnhumby-the-complete-journey/"
)

for filename in ["transaction_data.csv", "causal_data.csv"]:
    path = os.path.join(base_path, filename)

    df = pd.read_csv(path, nrows=5)

    print(f"\n{'=' * 45}")
    print(filename)
    print("Columns:", df.columns.tolist())
    display(df.head())



transaction_data.csv
Columns: ['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0,0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0,0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0,0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0,0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0,0



causal_data.csv
Columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A


In [3]:

import pandas as pd
import numpy as np

base_path = "/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/"

transactions = pd.read_csv(base_path + "transaction_data.csv")
promotions = pd.read_csv(base_path + "causal_data.csv")

print("Transactions shape:", transactions.shape)
print("Promotions shape:", promotions.shape)



Transactions shape: (2595732, 12)
Promotions shape: (36786524, 5)


In [4]:

def inspect_data(df, name):
    print(f"\n{'=' * 50}")
    print(name)
    print(f"{'=' * 50}")

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    display(df.isnull().sum().to_frame("missing_count"))

    print("\nExact duplicate rows:", df.duplicated().sum())

    print("\nNumerical summary:")
    display(df.describe().T)

inspect_data(transactions, "TRANSACTIONS")
inspect_data(promotions, "PROMOTIONS")



TRANSACTIONS

Data types:
household_key          int64
BASKET_ID              int64
DAY                    int64
PRODUCT_ID             int64
QUANTITY               int64
SALES_VALUE          float64
STORE_ID               int64
RETAIL_DISC          float64
TRANS_TIME             int64
WEEK_NO                int64
COUPON_DISC          float64
COUPON_MATCH_DISC    float64
dtype: object

Missing values:


,missing_count
household_key,0
BASKET_ID,0
DAY,0
PRODUCT_ID,0
QUANTITY,0
SALES_VALUE,0
STORE_ID,0
RETAIL_DISC,0
TRANS_TIME,0
WEEK_NO,0



Exact duplicate rows: 0

Numerical summary:


,count,mean,std,min,25%,50%,75%,max
household_key,2595732.0,1.271953e+03,7.260660e+02,1.000000e+00,6.560000e+02,1.272000e+03,1.913000e+03,2.500000e+03
BASKET_ID,2595732.0,3.402620e+10,4.711649e+09,2.698485e+10,3.040805e+10,3.276081e+10,4.012685e+10,4.230536e+10
DAY,2595732.0,3.887562e+02,1.897210e+02,1.000000e+00,2.290000e+02,3.900000e+02,5.530000e+02,7.110000e+02
PRODUCT_ID,2595732.0,2.891435e+06,3.837404e+06,2.567100e+04,9.174590e+05,1.028816e+06,1.133018e+06,1.831630e+07
QUANTITY,2595732.0,1.004286e+02,1.153436e+03,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,8.963800e+04
SALES_VALUE,2595732.0,3.104120e+00,4.182274e+00,0.000000e+00,1.290000e+00,2.000000e+00,3.490000e+00,8.400000e+02
STORE_ID,2595732.0,3.142673e+03,8.937113e+03,1.000000e+00,3.300000e+02,3.720000e+02,4.220000e+02,3.428000e+04
RETAIL_DISC,2595732.0,-5.387054e-01,1.249191e+00,-1.800000e+02,-6.900000e-01,-1.000000e-02,0.000000e+00,3.990000e+00
TRANS_TIME,2595732.0,1.561586e+03,3.998378e+02,0.000000e+00,1.308000e+03,1.613000e+03,1.843000e+03,2.359000e+03
WEEK_NO,2595732.0,5.622150e+01,2.710223e+01,1.000000e+00,3.300000e+01,5.600000e+01,8.000000e+01,1.020000e+02



PROMOTIONS

Data types:
PRODUCT_ID     int64
STORE_ID       int64
WEEK_NO        int64
display       object
mailer        object
dtype: object

Missing values:


,missing_count
PRODUCT_ID,0
STORE_ID,0
WEEK_NO,0
display,0
mailer,0



Exact duplicate rows: 0

Numerical summary:


,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,36786524.0,3.512237e+06,4.094819e+06,26190.0,928850.0,1312148.0,5730125.0,18244453.0
STORE_ID,36786524.0,3.234151e+03,9.106904e+03,286.0,329.0,369.0,421.0,34280.0
WEEK_NO,36786524.0,5.529931e+01,2.696101e+01,9.0,32.0,56.0,78.0,101.0


In [5]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

print("Promotion rows:", len(promotions))

print(
    "Unique promotion key combinations:",
    promotions[join_keys].drop_duplicates().shape[0]
)

print(
    "Duplicate promotion key combinations:",
    promotions.duplicated(subset=join_keys).sum()
)

print("\nDisplay value counts:")
print(promotions["display"].value_counts(dropna=False))

print("\nMailer value counts:")
print(promotions["mailer"].value_counts(dropna=False))


Promotion rows: 36786524
Unique promotion key combinations: 36771279
Duplicate promotion key combinations: 15245

Display value counts:
display
0    21038745
9     2699467
5     2575289
7     2362118
3     2073738
6     1816021
2     1812840
1     1102141
A      713180
4      592985
Name: count, dtype: int64

Mailer value counts:
mailer
A    17106789
0    11534183
D     4467453
H     1560395
F     1077549
J      306924
L      301327
C      291059
X      120823
Z       19453
P         569
Name: count, dtype: int64


In [6]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Keep only rows whose join-key combination occurs more than once
duplicate_mask = promotions.duplicated(
    subset=join_keys,
    keep=False
)

duplicate_promotions = promotions.loc[duplicate_mask].sort_values(
    join_keys
)

print("Rows involved in duplicate key combinations:",
      len(duplicate_promotions))

print("\nSample duplicate records:")
display(duplicate_promotions.head(20))

# Count distinct display/mailer combinations per key
value_combinations = (
    duplicate_promotions
    .groupby(join_keys)[["display", "mailer"]]
    .nunique()
)

print("\nKeys with conflicting display or mailer values:")
display(value_combinations[
    (value_combinations["display"] > 1) |
    (value_combinations["mailer"] > 1)
].head(20))


Rows involved in duplicate key combinations: 30490

Sample duplicate records:


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
45594,27658,361,59,0,D
45595,27658,361,59,6,D
45604,27658,361,76,0,D
45605,27658,361,76,6,D
49368,27658,34280,89,0,D
49369,27658,34280,89,6,D
49371,27658,34280,91,0,D
49372,27658,34280,91,6,D
49373,27658,34280,92,0,D
49374,27658,34280,92,6,D



Keys with conflicting display or mailer values:


display  mailer
PRODUCT_ID STORE_ID WEEK_NO                 
27658      361      59             2       1
                    76             2       1
           34280    89             2       1
                    91             2       1
                    92             2       1
                    93             2       1
                    94             2       1
                    95             2       1
                    98             2       1
28897      361      50             2       1
                    52             2       1
                    55             2       1
                    56             2       1
                    69             2       1
                    70             2       1
                    74             2       1
                    80             2       1
           34280    95             2       1
28966      361      48             2       1
                    59             2       1

In [7]:
join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Count distinct display/mailer combinations for each key
combinations = (
    promotions
    .drop_duplicates(join_keys + ["display", "mailer"])
    .groupby(join_keys, sort=False)
    .size()
)

print("Unique combinations per key:")
print(combinations.value_counts().sort_index())

Unique combinations per key:
1    36756034
2       15245
Name: count, dtype: int64


In [8]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Examine distinct values within each key
variation = (
    promotions.groupby(join_keys, sort=False)
    .agg(
        display_variants=("display", "nunique"),
        mailer_variants=("mailer", "nunique")
    )
)

print("Keys with multiple display values:",
      (variation["display_variants"] > 1).sum())

print("Keys with multiple mailer values:",
      (variation["mailer_variants"] > 1).sum())

print("Keys with both multiple display and mailer values:",
      (
          (variation["display_variants"] > 1) &
          (variation["mailer_variants"] > 1)
      ).sum())


Keys with multiple display values: 15208
Keys with multiple mailer values: 105
Keys with both multiple display and mailer values: 68


In [9]:

print("Display categories:")
print(sorted(promotions["display"].unique(), key=str))

print("\nMailer categories:")
print(sorted(promotions["mailer"].unique(), key=str))

print("\nDisplay and mailer combinations:")
display(
    promotions.groupby(["display", "mailer"], dropna=False)
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
    .head(25)
)


Display categories:
['0', '1', '2', '3', '4', '5', '6', '7', '9', 'A']

Mailer categories:
['0', 'A', 'C', 'D', 'F', 'H', 'J', 'L', 'P', 'X', 'Z']

Display and mailer combinations:


,display,mailer,record_count
0,0,A,14284146
2,0,D,3607593
86,9,0,2398879
53,5,0,1685025
75,7,0,1618969
64,6,0,1546161
32,3,0,1304402
4,0,H,1292966
21,2,0,1126383
3,0,F,1005296


In [10]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Count the number of distinct display and mailer codes per key.
# Built-in nunique is generally faster than groupby + lambda.
promotion_features = (
    promotions.groupby(join_keys, sort=False)
    .agg(
        display_code_count=("display", "nunique"),
        mailer_code_count=("mailer", "nunique")
    )
    .reset_index()
)

print("Original promotion rows:", len(promotions))
print("Aggregated promotion rows:", len(promotion_features))
print(
    "Duplicate keys:",
    promotion_features.duplicated(subset=join_keys).sum()
)

display(promotion_features.head())


Original promotion rows: 36786524
Aggregated promotion rows: 36771279
Duplicate keys: 0


,PRODUCT_ID,STORE_ID,WEEK_NO,display_code_count,mailer_code_count
0,26190,286,70,1,1
1,26190,288,70,1,1
2,26190,289,70,1,1
3,26190,292,70,1,1
4,26190,293,70,1,1


In [11]:

from pathlib import Path

output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

promotion_features.to_csv(
    output_dir / "promotion_features.csv",
    index=False
)

print("Saved promotion_features.csv")


Saved promotion_features.csv


In [12]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Validate key uniqueness before checking coverage
assert not promotion_features.duplicated(
    subset=join_keys
).any()

# Check match coverage using only the three join columns
transaction_keys = transactions[join_keys]

merge_check = transaction_keys.merge(
    promotion_features[join_keys],
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Transaction rows checked:", len(merge_check))
print("\nPromotion match status:")
print(merge_check["_merge"].value_counts())

del merge_check, transaction_keys


Transaction rows checked: 2595732

Promotion match status:
_merge
left_only     2031958
both           563774
right_only          0
Name: count, dtype: int64


In [13]:

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

for col in join_keys:
    transaction_values = set(transactions[col].unique())
    promotion_values = set(promotions[col].unique())

    print(f"\n{col}")
    print("Unique transaction values:", len(transaction_values))
    print("Unique promotion values:", len(promotion_values))
    print(
        "Values in both datasets:",
        len(transaction_values & promotion_values)
    )
    print(
        "Transaction values absent from promotions:",
        len(transaction_values - promotion_values)
    )



PRODUCT_ID
Unique transaction values: 92339
Unique promotion values: 68377
Values in both datasets: 68377
Transaction values absent from promotions: 23962

STORE_ID
Unique transaction values: 582
Unique promotion values: 115
Values in both datasets: 115
Transaction values absent from promotions: 467

WEEK_NO
Unique transaction values: 102
Unique promotion values: 93
Values in both datasets: 93
Transaction values absent from promotions: 9


In [14]:

transaction_keys = transactions[join_keys].drop_duplicates()

unmatched_keys = transaction_keys.merge(
    promotion_features[join_keys],
    on=join_keys,
    how="left",
    validate="one_to_one",
    indicator=True
)

unmatched_keys = unmatched_keys[
    unmatched_keys["_merge"] == "left_only"
]

print("Distinct unmatched transaction keys:", len(unmatched_keys))
display(unmatched_keys.head(10))

del transaction_keys, unmatched_keys


Distinct unmatched transaction keys: 1887464


,PRODUCT_ID,STORE_ID,WEEK_NO,_merge
0,1004906,364,1,left_only
1,1033142,364,1,left_only
2,1036325,364,1,left_only
3,1082185,364,1,left_only
4,8160430,364,1,left_only
5,826249,364,1,left_only
6,1043142,364,1,left_only
7,1085983,364,1,left_only
8,1102651,364,1,left_only
9,6423775,364,1,left_only


In [15]:

print("Transaction week range:",
      transactions["WEEK_NO"].min(),
      "to",
      transactions["WEEK_NO"].max())

print("\nPromotion week range:",
      promotions["WEEK_NO"].min(),
      "to",
      promotions["WEEK_NO"].max())

print("\nWeeks in transactions but not promotions:")
print(sorted(
    set(transactions["WEEK_NO"].unique())
    - set(promotions["WEEK_NO"].unique())
))

print("\nStores in transactions but not promotions (first 30):")
print(sorted(
    set(transactions["STORE_ID"].unique())
    - set(promotions["STORE_ID"].unique())
)[:30])


Transaction week range: 1 to 102

Promotion week range: 9 to 101

Weeks in transactions but not promotions:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(102)]

Stores in transactions but not promotions (first 30):
[np.int64(1), np.int64(2), np.int64(12), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(26), np.int64(27), np.int64(37), np.int64(42), np.int64(45), np.int64(46), np.int64(53), np.int64(55), np.int64(62), np.int64(65), np.int64(77), np.int64(81), np.int64(84), np.int64(86), np.int64(105), np.int64(107), np.int64(108), np.int64(109), np.int64(110), np.int64(112), np.int64(115), np.int64(121), np.int64(128)]


In [16]:

import os
import pandas as pd

base_path = "/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/"

for filename in ["campaign_desc.csv", "campaign_table.csv"]:
    path = os.path.join(base_path, filename)

    print(f"\n{'=' * 50}")
    print(filename)

    df = pd.read_csv(path, nrows=5)

    print("Columns:", df.columns.tolist())
    display(df.head())



campaign_desc.csv
Columns: ['DESCRIPTION', 'CAMPAIGN', 'START_DAY', 'END_DAY']


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684



campaign_table.csv
Columns: ['DESCRIPTION', 'household_key', 'CAMPAIGN']


,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [17]:

from pathlib import Path

output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

promotion_features.to_csv(
    output_dir / "promotion_features.csv",
    index=False
)

print("Saved promotion_features.csv")
print("Rows:", len(promotion_features))
print(
    "Duplicate keys:",
    promotion_features.duplicated(
        subset=["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
    ).sum()
)


Saved promotion_features.csv
Rows: 36771279
Duplicate keys: 0


In [18]:

columns = [
    "QUANTITY",
    "SALES_VALUE",
    "RETAIL_DISC",
    "COUPON_DISC",
    "COUPON_MATCH_DISC"
]

print("Transaction summary:")
display(transactions[columns].describe().T)

print("\nNonpositive quantities:")
print((transactions["QUANTITY"] <= 0).sum())

print("\nZero quantities:")
print((transactions["QUANTITY"] == 0).sum())

print("\nNegative sales values:")
print((transactions["SALES_VALUE"] < 0).sum())

print("\nZero sales values:")
print((transactions["SALES_VALUE"] == 0).sum())


Transaction summary:


,count,mean,std,min,25%,50%,75%,max
QUANTITY,2595732.0,100.428558,1153.436211,0.00,1.00,1.00,1.00,89638.00
SALES_VALUE,2595732.0,3.104120,4.182274,0.00,1.29,2.00,3.49,840.00
RETAIL_DISC,2595732.0,-0.538705,1.249191,-180.00,-0.69,-0.01,0.00,3.99
COUPON_DISC,2595732.0,-0.016416,0.216841,-55.93,0.00,0.00,0.00,0.00
COUPON_MATCH_DISC,2595732.0,-0.002919,0.039690,-7.70,0.00,0.00,0.00,0.00



Nonpositive quantities:
14466

Zero quantities:
14466

Negative sales values:
0

Zero sales values:
18850


In [19]:

import json
from pathlib import Path

output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    "transaction_rows": int(len(transactions)),
    "nonpositive_quantity_rows": int(
        (transactions["QUANTITY"] <= 0).sum()
    ),
    "zero_quantity_rows": int(
        (transactions["QUANTITY"] == 0).sum()
    ),
    "negative_sales_rows": int(
        (transactions["SALES_VALUE"] < 0).sum()
    ),
    "zero_sales_rows": int(
        (transactions["SALES_VALUE"] == 0).sum()
    ),
    "checked_columns": columns
}

with open(output_dir / "transaction_validation_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved transaction_validation_report.json")


Saved transaction_validation_report.json


In [20]:

print("Top 15 quantities:")
display(
    transactions.nlargest(15, "QUANTITY")[
        [
            "household_key", "BASKET_ID", "DAY",
            "PRODUCT_ID", "QUANTITY", "SALES_VALUE",
            "STORE_ID", "RETAIL_DISC", "COUPON_DISC"
        ]
    ]
)

print("\nRecords with zero quantity:")
display(
    transactions.loc[
        transactions["QUANTITY"] == 0,
        [
            "household_key", "BASKET_ID", "DAY",
            "PRODUCT_ID", "QUANTITY", "SALES_VALUE",
            "STORE_ID", "RETAIL_DISC", "COUPON_DISC"
        ]
    ].head(10)
)

print("\nRecords with zero sales:")
display(
    transactions.loc[
        transactions["SALES_VALUE"] == 0,
        [
            "household_key", "BASKET_ID", "DAY",
            "PRODUCT_ID", "QUANTITY", "SALES_VALUE",
            "STORE_ID", "RETAIL_DISC", "COUPON_DISC"
        ]
    ].head(10)
)


Top 15 quantities:


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,COUPON_DISC
1750942,630,34749153595,503,6534178,89638,250.00,384,-13.45,0.0
468356,2407,29392047893,181,6544236,85055,210.00,375,-8.51,0.0
481876,630,29484790880,185,6534178,61335,150.21,384,-6.13,0.0
166536,149,28210551971,103,6534178,51912,110.00,327,-5.19,0.0
1340882,193,32956767959,402,6534178,48073,121.10,310,-7.21,0.0
2472791,2133,41904760458,682,6534178,45475,100.00,433,-4.55,0.0
1560340,149,33768630428,456,6534178,41833,115.00,327,-4.18,0.0
376686,149,29035716247,158,6534178,41686,100.00,327,-4.17,0.0
156049,1406,28167562655,99,6544236,41485,85.00,327,-4.15,0.0
87625,107,27865225627,74,6534178,39365,72.00,329,-3.94,0.0



Records with zero quantity:


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,COUPON_DISC
97,744,26985165432,1,5978648,0,0.0,31582,0.0,0.00
128,1287,26985336468,1,5978648,0,0.0,304,0.0,0.00
249,2305,26996870743,2,5978656,0,0.0,414,0.0,-1.00
293,271,26997082949,2,5978656,0,0.0,329,0.0,-2.00
694,315,27008952267,3,957951,0,0.0,327,0.0,0.00
1089,14,27021203242,4,6463658,0,0.0,319,0.0,0.00
1649,2305,27031060487,5,5978648,0,0.0,414,0.0,0.00
1985,100,27045626628,6,5978648,0,0.0,368,0.0,0.00
2142,907,27045792138,6,1125278,0,0.0,313,0.0,0.00
2143,907,27045792138,6,5978656,0,0.0,313,0.0,-2.19



Records with zero sales:


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,COUPON_DISC
97,744,26985165432,1,5978648,0,0.0,31582,0.00,0.0
128,1287,26985336468,1,5978648,0,0.0,304,0.00,0.0
249,2305,26996870743,2,5978656,0,0.0,414,0.00,-1.0
293,271,26997082949,2,5978656,0,0.0,329,0.00,-2.0
694,315,27008952267,3,957951,0,0.0,327,0.00,0.0
766,2110,27009082349,3,837963,1,0.0,375,0.00,0.0
996,568,27021090189,4,951412,1,0.0,446,-1.48,0.0
1089,14,27021203242,4,6463658,0,0.0,319,0.00,0.0
1095,2110,27021248054,4,850115,1,0.0,375,-3.29,0.0
1113,2110,27021248054,4,976872,1,0.0,375,-2.49,0.0


In [21]:

print("Zero quantity and positive sales:")
print(
    (
        (transactions["QUANTITY"] == 0) &
        (transactions["SALES_VALUE"] > 0)
    ).sum()
)

print("\nPositive quantity and zero sales:")
print(
    (
        (transactions["QUANTITY"] > 0) &
        (transactions["SALES_VALUE"] == 0)
    ).sum()
)

print("\nQuantities greater than 100:")
print((transactions["QUANTITY"] > 100).sum())

print("\nQuantities greater than 1000:")
print((transactions["QUANTITY"] > 1000).sum())


Zero quantity and positive sales:
67

Positive quantity and zero sales:
4451

Quantities greater than 100:
23136

Quantities greater than 1000:
23101


In [22]:

from pathlib import Path
import json

output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    "transaction_rows": int(len(transactions)),
    "zero_quantity_rows": int(
        (transactions["QUANTITY"] == 0).sum()
    ),
    "zero_sales_rows": int(
        (transactions["SALES_VALUE"] == 0).sum()
    ),
    "zero_quantity_positive_sales": int(
        (
            (transactions["QUANTITY"] == 0) &
            (transactions["SALES_VALUE"] > 0)
        ).sum()
    ),
    "positive_quantity_zero_sales": int(
        (
            (transactions["QUANTITY"] > 0) &
            (transactions["SALES_VALUE"] == 0)
        ).sum()
    ),
    "quantity_above_100": int(
        (transactions["QUANTITY"] > 100).sum()
    ),
    "quantity_above_1000": int(
        (transactions["QUANTITY"] > 1000).sum()
    )
}

with open(output_dir / "transaction_validation_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Validation report saved.")


Validation report saved.


In [23]:

print("Quantity percentiles:")
display(
    transactions["QUANTITY"]
    .quantile([0.50, 0.90, 0.95, 0.99, 0.999, 1.00])
    .to_frame("QUANTITY")
)

print("\nLargest quantities and corresponding sales:")
display(
    transactions.nlargest(20, "QUANTITY")[
        ["PRODUCT_ID", "STORE_ID", "WEEK_NO",
         "QUANTITY", "SALES_VALUE", "RETAIL_DISC",
         "COUPON_DISC", "COUPON_MATCH_DISC"]
    ]
)

print("\nQuantity frequency for values above 1000:")
display(
    transactions.loc[
        transactions["QUANTITY"] > 1000, "QUANTITY"
    ].value_counts().head(15)
)

# Preserve the original dataset and columns.
transactions["UNIT_PRICE_PRELIM"] = np.nan

positive_quantity = transactions["QUANTITY"] > 0

transactions.loc[
    positive_quantity, "UNIT_PRICE_PRELIM"
] = (
    transactions.loc[positive_quantity, "SALES_VALUE"]
    / transactions.loc[positive_quantity, "QUANTITY"]
)

print("Total transaction rows:", len(transactions))
print(
    "Rows with a preliminary unit price:",
    transactions["UNIT_PRICE_PRELIM"].notna().sum()
)
print(
    "Rows without a preliminary unit price:",
    transactions["UNIT_PRICE_PRELIM"].isna().sum()
)

display(transactions["UNIT_PRICE_PRELIM"].describe())


Quantity percentiles:


,QUANTITY
0.500,1.000
0.900,2.000
0.950,3.000
0.990,10.000
0.999,16943.538
1.000,89638.000



Largest quantities and corresponding sales:


,PRODUCT_ID,STORE_ID,WEEK_NO,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC
1750942,6534178,384,73,89638,250.00,-13.45,0.0,0.0
468356,6544236,375,27,85055,210.00,-8.51,0.0,0.0
481876,6534178,384,27,61335,150.21,-6.13,0.0,0.0
166536,6534178,327,15,51912,110.00,-5.19,0.0,0.0
1340882,6534178,310,58,48073,121.10,-7.21,0.0,0.0
2472791,6534178,433,98,45475,100.00,-4.55,0.0,0.0
1560340,6534178,327,66,41833,115.00,-4.18,0.0,0.0
376686,6534178,327,23,41686,100.00,-4.17,0.0,0.0
156049,6544236,327,15,41485,85.00,-4.15,0.0,0.0
87625,6534178,329,11,39365,72.00,-3.94,0.0,0.0



Quantity frequency for values above 1000:


QUANTITY
9528     33
9095     31
9094     28
9526     25
10005    25
4547     24
4350     22
7146     22
10004    22
8699     21
4632     21
10006    20
8337     20
4548     20
4549     20
Name: count, dtype: int64

Total transaction rows: 2595732
Rows with a preliminary unit price: 2581266
Rows without a preliminary unit price: 14466


count    2.581266e+06
mean     2.436950e+00
std      2.720470e+00
min      0.000000e+00
25%      1.000000e+00
50%      1.920000e+00
75%      2.990000e+00
max      4.999900e+02
Name: UNIT_PRICE_PRELIM, dtype: float64

In [24]:

price = transactions["UNIT_PRICE_PRELIM"]

print("Zero preliminary prices:",
      (price == 0).sum())

print("Positive preliminary prices:",
      (price > 0).sum())

print("Negative preliminary prices:",
      (price < 0).sum())

print("\nPrice percentiles:")
display(
    price.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00])
    .to_frame("UNIT_PRICE_PRELIM")
)

print("\nSample records with zero preliminary price:")
display(
    transactions.loc[
        price == 0,
        ["PRODUCT_ID", "STORE_ID", "WEEK_NO",
         "QUANTITY", "SALES_VALUE", "RETAIL_DISC",
         "COUPON_DISC", "COUPON_MATCH_DISC"]
    ].head(10)
)


Zero preliminary prices: 4451
Positive preliminary prices: 2576815
Negative preliminary prices: 0

Price percentiles:


,UNIT_PRICE_PRELIM
0.01,0.002859
0.05,0.400000
0.25,1.000000
0.50,1.920000
0.75,2.990000
0.95,6.170000
0.99,11.990000
1.00,499.990000



Sample records with zero preliminary price:


,PRODUCT_ID,STORE_ID,WEEK_NO,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC
766,837963,375,1,1,0.0,0.00,0.0,0.0
996,951412,446,1,1,0.0,-1.48,0.0,0.0
1095,850115,375,1,1,0.0,-3.29,0.0,0.0
1113,976872,375,1,1,0.0,-2.49,0.0,0.0
1161,940108,352,1,1,0.0,-1.19,0.0,0.0
1863,1114465,298,1,1,0.0,-1.39,0.0,0.0
2611,961785,433,2,1,0.0,-2.99,0.0,0.0
3054,6632482,324,2,1,0.0,-4.99,0.0,0.0
3195,991268,372,2,1,0.0,0.00,0.0,0.0
3356,905181,321,2,1,0.0,-1.99,0.0,0.0


In [25]:

print("Top 20 preliminary unit prices:")

display(
    transactions.nlargest(20, "UNIT_PRICE_PRELIM")[
        [
            "PRODUCT_ID",
            "STORE_ID",
            "WEEK_NO",
            "QUANTITY",
            "SALES_VALUE",
            "RETAIL_DISC",
            "COUPON_DISC",
            "COUPON_MATCH_DISC",
            "UNIT_PRICE_PRELIM"
        ]
    ]
)

zero_price = transactions[
    transactions["UNIT_PRICE_PRELIM"] == 0
]

print("Zero-price records:", len(zero_price))

print("\nRetail discount distribution:")
display(zero_price["RETAIL_DISC"].describe())

print("\nZero-price records with nonzero retail discount:")
print((zero_price["RETAIL_DISC"] != 0).sum())

print("\nCoupon discount distribution:")
display(zero_price["COUPON_DISC"].describe())


Top 20 preliminary unit prices:


,PRODUCT_ID,STORE_ID,WEEK_NO,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC,UNIT_PRICE_PRELIM
1152895,13212959,361,51,1,499.99,-50.0,0.0,0.0,499.99
547657,1089093,323,29,1,455.00,0.0,0.0,0.0,455.00
526495,1089093,343,29,1,430.00,0.0,0.0,0.0,430.00
3600,1089093,372,2,1,400.00,0.0,0.0,0.0,400.00
2352647,1089093,31782,94,1,380.00,0.0,0.0,0.0,380.00
1366400,13506935,356,59,1,329.99,0.0,0.0,0.0,329.99
1338386,13213287,343,58,1,299.99,0.0,0.0,0.0,299.99
1484604,13381576,365,63,1,299.99,0.0,0.0,0.0,299.99
1085442,12484608,412,49,3,840.00,0.0,0.0,0.0,280.00
655985,1089093,323,34,2,505.00,0.0,0.0,0.0,252.50


Zero-price records: 4451

Retail discount distribution:


count    4451.000000
mean       -3.307776
std         2.369308
min       -39.950000
25%        -3.990000
50%        -3.290000
75%        -2.045000
max         0.000000
Name: RETAIL_DISC, dtype: float64


Zero-price records with nonzero retail discount:
3748

Coupon discount distribution:


count    4451.000000
mean       -0.010825
std         0.153998
min        -7.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.000000
Name: COUPON_DISC, dtype: float64

In [26]:

zero_sales = transactions[
    (transactions["QUANTITY"] > 0) &
    (transactions["SALES_VALUE"] == 0)
]

print("Zero-sales records:", len(zero_sales))

print("\nRetail discount summary:")
display(
    zero_sales.groupby("RETAIL_DISC")
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("record_count")
    .to_frame()
)

print("\nSample zero-sales records:")
display(
    zero_sales[
        [
            "PRODUCT_ID",
            "STORE_ID",
            "WEEK_NO",
            "QUANTITY",
            "SALES_VALUE",
            "RETAIL_DISC",
            "COUPON_DISC",
            "COUPON_MATCH_DISC"
        ]
    ].head(15)
)


Zero-sales records: 4451

Retail discount summary:


,record_count
RETAIL_DISC,
-3.99,857
0.00,703
-2.99,430
-2.59,191
-3.29,190
-5.99,159
-0.99,130
-1.99,125
-3.49,110



Sample zero-sales records:


,PRODUCT_ID,STORE_ID,WEEK_NO,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC
766,837963,375,1,1,0.0,0.00,0.0,0.0
996,951412,446,1,1,0.0,-1.48,0.0,0.0
1095,850115,375,1,1,0.0,-3.29,0.0,0.0
1113,976872,375,1,1,0.0,-2.49,0.0,0.0
1161,940108,352,1,1,0.0,-1.19,0.0,0.0
1863,1114465,298,1,1,0.0,-1.39,0.0,0.0
2611,961785,433,2,1,0.0,-2.99,0.0,0.0
3054,6632482,324,2,1,0.0,-4.99,0.0,0.0
3195,991268,372,2,1,0.0,0.00,0.0,0.0
3356,905181,321,2,1,0.0,-1.99,0.0,0.0


In [27]:

# Preserve all original transaction columns.
transactions["UNIT_PRICE_PRELIM"] = np.nan

positive_quantity = transactions["QUANTITY"] > 0

transactions.loc[
    positive_quantity, "UNIT_PRICE_PRELIM"
] = (
    transactions.loc[positive_quantity, "SALES_VALUE"]
    / transactions.loc[positive_quantity, "QUANTITY"]
)

# Diagnostic flags; these do not delete or modify observations.
transactions["ZERO_QUANTITY_FLAG"] = (
    transactions["QUANTITY"] == 0
).astype("int8")

transactions["ZERO_SALES_FLAG"] = (
    transactions["SALES_VALUE"] == 0
).astype("int8")

transactions["EXTREME_QUANTITY_FLAG"] = (
    transactions["QUANTITY"] > 1000
).astype("int8")

print("Rows preserved:", len(transactions))
print("\nFlag counts:")
display(
    transactions[
        [
            "ZERO_QUANTITY_FLAG",
            "ZERO_SALES_FLAG",
            "EXTREME_QUANTITY_FLAG"
        ]
    ].sum().to_frame("count")
)

print("\nPreliminary unit-price summary:")
display(transactions["UNIT_PRICE_PRELIM"].describe())


Rows preserved: 2595732

Flag counts:


,count
ZERO_QUANTITY_FLAG,14466
ZERO_SALES_FLAG,18850
EXTREME_QUANTITY_FLAG,23101



Preliminary unit-price summary:


count    2.581266e+06
mean     2.436950e+00
std      2.720470e+00
min      0.000000e+00
25%      1.000000e+00
50%      1.920000e+00
75%      2.990000e+00
max      4.999900e+02
Name: UNIT_PRICE_PRELIM, dtype: float64

In [28]:

from pathlib import Path

output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

feature_path = output_dir / "transactions_preliminary_features.csv"

transactions.to_csv(feature_path, index=False)

print("Saved:", feature_path)
print("Rows:", len(transactions))
print("Columns:", len(transactions.columns))


Saved: /kaggle/working/causal_variable_selection/transactions_preliminary_features.csv
Rows: 2595732
Columns: 16


In [29]:
# Check transaction and promotion coverage by week

transaction_weeks = set(transactions["WEEK_NO"].unique())
promotion_weeks = set(promotions["WEEK_NO"].unique())

print("Transaction week range:",
      transactions["WEEK_NO"].min(),
      "to",
      transactions["WEEK_NO"].max())

print("Promotion week range:",
      promotions["WEEK_NO"].min(),
      "to",
      promotions["WEEK_NO"].max())

print("Weeks in transactions but not promotions:",
      sorted(transaction_weeks - promotion_weeks))

print("Weeks in promotions but not transactions:",
      sorted(promotion_weeks - transaction_weeks))

Transaction week range: 1 to 102
Promotion week range: 9 to 101
Weeks in transactions but not promotions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(102)]
Weeks in promotions but not transactions: []


In [30]:
# Check promotion-key coverage by week

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Unique product-store-week combinations in promotion data
promotion_keys = promotions[join_keys].drop_duplicates()

# Unique transaction keys, with the transaction week retained
transaction_keys = transactions[join_keys].drop_duplicates()

# Match each transaction key against promotion keys
weekly_coverage = transaction_keys.merge(
    promotion_keys.assign(PROMOTION_KEY_FOUND=1),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

weekly_coverage["PROMOTION_KEY_FOUND"] = (
    weekly_coverage["PROMOTION_KEY_FOUND"].fillna(0).astype("int8")
)

weekly_report = (
    weekly_coverage.groupby("WEEK_NO")
    .agg(
        unique_transaction_keys=("PRODUCT_ID", "size"),
        matched_keys=("PROMOTION_KEY_FOUND", "sum")
    )
    .reset_index()
)

weekly_report["unmatched_keys"] = (
    weekly_report["unique_transaction_keys"]
    - weekly_report["matched_keys"]
)

weekly_report["match_percentage"] = (
    100 * weekly_report["matched_keys"]
    / weekly_report["unique_transaction_keys"]
).round(2)

display(weekly_report)

,WEEK_NO,unique_transaction_keys,matched_keys,unmatched_keys,match_percentage
0,1,1855,0,1855,0.00
1,2,3597,0,3597,0.00
2,3,4630,0,4630,0.00
3,4,5175,0,5175,0.00
4,5,6866,0,6866,0.00
...,...,...,...,...,...
97,98,28377,5932,22445,20.90
98,99,29365,6528,22837,22.23
99,100,26682,5196,21486,19.47
100,101,24440,4506,19934,18.44


In [31]:
join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# One row per distinct promotion key
promotion_keys = promotions[join_keys].drop_duplicates()

# One row per distinct transaction key
transaction_keys = transactions[join_keys].drop_duplicates()

# Check whether each transaction key exists in promotions
weekly_coverage = transaction_keys.merge(
    promotion_keys.assign(PROMOTION_KEY_FOUND=1),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

weekly_coverage["PROMOTION_KEY_FOUND"] = (
    weekly_coverage["PROMOTION_KEY_FOUND"]
    .fillna(0)
    .astype("int8")
)

# Aggregate by week
weekly_report = (
    weekly_coverage.groupby("WEEK_NO")
    .agg(
        unique_transaction_keys=("PRODUCT_ID", "size"),
        matched_keys=("PROMOTION_KEY_FOUND", "sum")
    )
    .reset_index()
)

weekly_report["unmatched_keys"] = (
    weekly_report["unique_transaction_keys"]
    - weekly_report["matched_keys"]
)

weekly_report["match_percentage"] = (
    100 * weekly_report["matched_keys"]
    / weekly_report["unique_transaction_keys"]
).round(2)

# Display selected weeks for easy verification
display(
    weekly_report[
        weekly_report["WEEK_NO"].isin([0, 1, 8, 9, 50, 97, 100, 101, 102])
    ].sort_values("WEEK_NO")
)

# Verify the counts are logically consistent
assert (
    weekly_report["matched_keys"]
    <= weekly_report["unique_transaction_keys"]
).all()

assert (
    weekly_report["unmatched_keys"]
    == weekly_report["unique_transaction_keys"]
       - weekly_report["matched_keys"]
).all()

,WEEK_NO,unique_transaction_keys,matched_keys,unmatched_keys,match_percentage
0,1,1855,0,1855,0.00
7,8,9908,0,9908,0.00
8,9,9967,2326,7641,23.34
49,50,26954,5464,21490,20.27
96,97,26653,6222,20431,23.34
99,100,26682,5196,21486,19.47
100,101,24440,4506,19934,18.44
101,102,20901,0,20901,0.00


In [32]:
print("Transaction WEEK_NO values:")
print(sorted(transactions["WEEK_NO"].dropna().unique())[:12])
print("...")
print(sorted(transactions["WEEK_NO"].dropna().unique())[-5:])

print("\nWeekly coverage report:")
print(
    weekly_report[
        weekly_report["WEEK_NO"].isin([0, 1, 7, 8, 9, 49, 50, 96, 97, 100, 101, 102])
    ].sort_values("WEEK_NO").to_string(index=False)
)

Transaction WEEK_NO values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
...
[np.int64(98), np.int64(99), np.int64(100), np.int64(101), np.int64(102)]

Weekly coverage report:
 WEEK_NO  unique_transaction_keys  matched_keys  unmatched_keys  match_percentage
       1                     1855             0            1855              0.00
       7                     8620             0            8620              0.00
       8                     9908             0            9908              0.00
       9                     9967          2326            7641             23.34
      49                    26270          6101           20169             23.22
      50                    26954          5464           21490             20.27
      96                    25786          4938           20848             19.15
      97                    26653          6222      

In [33]:
# Compare store coverage between transactions and promotions

transaction_stores = set(transactions["STORE_ID"].unique())
promotion_stores = set(promotions["STORE_ID"].unique())

print("Unique transaction stores:", len(transaction_stores))
print("Unique promotion stores:", len(promotion_stores))

print(
    "Transaction stores absent from promotions:",
    len(transaction_stores - promotion_stores)
)

print(
    "Promotion stores absent from transactions:",
    len(promotion_stores - transaction_stores)
)

print("\nSample transaction stores absent from promotions:")
print(sorted(transaction_stores - promotion_stores)[:20])

Unique transaction stores: 582
Unique promotion stores: 115
Transaction stores absent from promotions: 467
Promotion stores absent from transactions: 0

Sample transaction stores absent from promotions:
[np.int64(1), np.int64(2), np.int64(12), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(26), np.int64(27), np.int64(37), np.int64(42), np.int64(45), np.int64(46), np.int64(53), np.int64(55), np.int64(62), np.int64(65), np.int64(77), np.int64(81), np.int64(84)]


In [34]:
# Restrict transactions to weeks covered by promotions
transactions_in_period = transactions[
    transactions["WEEK_NO"].between(9, 101)
]

covered_stores = transactions_in_period["STORE_ID"].isin(
    promotion_stores
)

print(
    "Transactions in weeks 9-101:",
    len(transactions_in_period)
)

print(
    "Transactions in covered stores:",
    int(covered_stores.sum())
)

print(
    "Percentage in covered stores:",
    round(covered_stores.mean() * 100, 2), "%"
)

Transactions in weeks 9-101: 2522099
Transactions in covered stores: 2486644
Percentage in covered stores: 98.59 %


In [35]:
join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Unique keys available in the promotion dataset
promotion_keys = promotions[join_keys].drop_duplicates()

# Keep transactions within promotion weeks and covered stores
eligible_transactions = transactions[
    transactions["WEEK_NO"].between(9, 101)
    & transactions["STORE_ID"].isin(promotion_stores)
]

# Get unique transaction keys for a fair comparison
eligible_keys = eligible_transactions[join_keys].drop_duplicates()

# Match keys against promotion data
eligible_coverage = eligible_keys.merge(
    promotion_keys.assign(PROMOTION_KEY_FOUND=1),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

matched = eligible_coverage["PROMOTION_KEY_FOUND"].notna().sum()
total = len(eligible_coverage)
unmatched = total - matched

print("Eligible unique transaction keys:", total)
print("Matched keys:", matched)
print("Unmatched keys:", unmatched)
print("Match percentage:", round(matched / total * 100, 2), "%")

Eligible unique transaction keys: 2265861
Matched keys: 483320
Unmatched keys: 1782541
Match percentage: 21.33 %


In [37]:
join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# Unique eligible transaction keys
eligible_keys = eligible_transactions[join_keys].drop_duplicates()

# Sets of products, stores, and store-week combinations in promotions
promo_products = set(promotions["PRODUCT_ID"].unique())

promo_store_weeks = promotions[
    ["STORE_ID", "WEEK_NO"]
].drop_duplicates()

# Check whether product and store-week exist in promotion data
eligible_keys = eligible_keys.copy()

eligible_keys["PRODUCT_EXISTS"] = (
    eligible_keys["PRODUCT_ID"].isin(promo_products)
)

eligible_keys = eligible_keys.merge(
    promo_store_weeks.assign(STORE_WEEK_EXISTS=True),
    on=["STORE_ID", "WEEK_NO"],
    how="left",
    validate="many_to_one"
)

eligible_keys["STORE_WEEK_EXISTS"] = (
    eligible_keys["STORE_WEEK_EXISTS"].fillna(False)
)

# Check the exact product-store-week match
eligible_keys = eligible_keys.merge(
    promotion_keys.assign(EXACT_KEY_EXISTS=True),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

eligible_keys["EXACT_KEY_EXISTS"] = (
    eligible_keys["EXACT_KEY_EXISTS"].fillna(False)
)

print("Total eligible keys:", len(eligible_keys))
print("\nProduct exists in promotions:")
print(eligible_keys["PRODUCT_EXISTS"].value_counts())

print("\nStore-week exists in promotions:")
print(eligible_keys["STORE_WEEK_EXISTS"].value_counts())

print("\nExact product-store-week key exists:")
print(eligible_keys["EXACT_KEY_EXISTS"].value_counts())

print("\nKeys where product and store-week exist, but exact key does not:")
print(
    (
        eligible_keys["PRODUCT_EXISTS"]
        & eligible_keys["STORE_WEEK_EXISTS"]
        & ~eligible_keys["EXACT_KEY_EXISTS"]
    ).sum()
)

/tmp/ipykernel_130/522983203.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eligible_keys["STORE_WEEK_EXISTS"].fillna(False)


Total eligible keys: 2265861

Product exists in promotions:
PRODUCT_EXISTS
True     2054411
False     211450
Name: count, dtype: int64

Store-week exists in promotions:
STORE_WEEK_EXISTS
True     2265786
False         75
Name: count, dtype: int64

Exact product-store-week key exists:
EXACT_KEY_EXISTS
False    1782541
True      483320
Name: count, dtype: int64

Keys where product and store-week exist, but exact key does not:
1571039


/tmp/ipykernel_130/522983203.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eligible_keys["EXACT_KEY_EXISTS"].fillna(False)


In [38]:
# Step 2A: Inspect display and mailer codes

print("DISPLAY CODE COUNTS:")
print(promotions["display"].value_counts(dropna=False).sort_index())

print("\nMAILER CODE COUNTS:")
print(promotions["mailer"].value_counts(dropna=False).sort_index())

# Step 2B: Inspect products represented per store-week

store_week_coverage = (
    promotions.groupby(["STORE_ID", "WEEK_NO"])
    .agg(
        unique_products=("PRODUCT_ID", "nunique"),
        promotion_rows=("PRODUCT_ID", "size")
    )
    .reset_index()
)

print("\nSTORE-WEEK COVERAGE SUMMARY:")
print(
    store_week_coverage["unique_products"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99])
)

print("\nSTORE-WEEK EXAMPLES:")
display(store_week_coverage.head(10))

DISPLAY CODE COUNTS:
display
0    21038745
1     1102141
2     1812840
3     2073738
4      592985
5     2575289
6     1816021
7     2362118
9     2699467
A      713180
Name: count, dtype: int64

MAILER CODE COUNTS:
mailer
0    11534183
A    17106789
C      291059
D     4467453
F     1077549
H     1560395
J      306924
L      301327
P         569
X      120823
Z       19453
Name: count, dtype: int64

STORE-WEEK COVERAGE SUMMARY:
count    10047.000000
mean      3659.926247
std        778.983761
min        262.000000
25%       3172.000000
50%       3664.000000
75%       4096.000000
90%       4591.400000
99%       5754.160000
max       6921.000000
Name: unique_products, dtype: float64

STORE-WEEK EXAMPLES:


,STORE_ID,WEEK_NO,unique_products,promotion_rows
0,286,9,2095,2095
1,286,10,1820,1820
2,286,11,2944,2944
3,286,12,1911,1911
4,286,13,2178,2178
5,286,14,2047,2047
6,286,15,2031,2031
7,286,16,3246,3246
8,286,17,2072,2072
9,286,18,1833,1833


In [39]:
# Step 3: Compare product coverage within each store-week

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
store_week_keys = ["STORE_ID", "WEEK_NO"]

# Restrict transactions to weeks and stores covered by promotions
eligible_transactions = transactions[
    transactions["WEEK_NO"].between(9, 101)
    & transactions["STORE_ID"].isin(promotion_stores)
]

# Count distinct products in transactions per store-week
transaction_store_week = (
    eligible_transactions.groupby(store_week_keys)["PRODUCT_ID"]
    .nunique()
    .rename("transaction_products")
    .reset_index()
)

# Count distinct products in promotions per store-week
promotion_store_week = (
    promotions.groupby(store_week_keys)["PRODUCT_ID"]
    .nunique()
    .rename("promotion_products")
    .reset_index()
)

# Compare the two datasets
store_week_comparison = transaction_store_week.merge(
    promotion_store_week,
    on=store_week_keys,
    how="left",
    validate="one_to_one"
)

store_week_comparison["promotion_products"] = (
    store_week_comparison["promotion_products"].fillna(0)
)

store_week_comparison["product_coverage_pct"] = (
    100
    * store_week_comparison["transaction_products"]
    / store_week_comparison["promotion_products"].replace(0, np.nan)
).round(2)

print("Store-week combinations compared:", len(store_week_comparison))

print("\nProduct coverage percentage summary:")
print(store_week_comparison["product_coverage_pct"].describe())

print("\nLowest-coverage store-week combinations:")
display(
    store_week_comparison
    .sort_values("product_coverage_pct")
    .head(10)
)

print("\nHighest-coverage store-week combinations:")
display(
    store_week_comparison
    .sort_values("product_coverage_pct", ascending=False)
    .head(10)
)

Store-week combinations compared: 10045

Product coverage percentage summary:
count    10038.000000
mean         6.302880
std          3.553841
min          0.020000
25%          3.790000
50%          5.770000
75%          8.130000
max         32.490000
Name: product_coverage_pct, dtype: float64

Lowest-coverage store-week combinations:


,STORE_ID,WEEK_NO,transaction_products,promotion_products,product_coverage_pct
9968,34007,78,1,4669.0,0.02
3831,352,31,1,3177.0,0.03
3809,352,9,2,3196.0,0.06
4951,366,70,1,1515.0,0.07
4926,366,44,2,1663.0,0.12
5675,379,52,3,2238.0,0.13
9980,34007,90,8,5328.0,0.15
5709,379,89,4,2459.0,0.16
8506,442,11,8,5011.0,0.16
9960,34007,69,7,4177.0,0.17



Highest-coverage store-week combinations:


,STORE_ID,WEEK_NO,transaction_products,promotion_products,product_coverage_pct
5826,381,21,651,2004.0,32.49
9901,33923,50,82,262.0,31.30
7009,406,88,635,2428.0,26.15
9884,32004,88,428,1643.0,26.05
4997,367,23,746,2909.0,25.64
5057,367,83,830,3357.0,24.72
3563,343,42,615,2521.0,24.40
7019,406,98,756,3103.0,24.36
4473,361,42,570,2374.0,24.01
5063,367,89,843,3515.0,23.98


In [40]:
# Step 4: Calculate actual product matching coverage

join_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
store_week_keys = ["STORE_ID", "WEEK_NO"]

# One row per unique eligible transaction key
eligible_keys = eligible_transactions[join_keys].drop_duplicates()

# One row per unique promotion key
promotion_keys = promotions[join_keys].drop_duplicates()

# Check exact product-store-week matches
key_matches = eligible_keys.merge(
    promotion_keys.assign(MATCHED=1),
    on=join_keys,
    how="left",
    validate="one_to_one"
)

key_matches["MATCHED"] = (
    key_matches["MATCHED"].fillna(0).astype("int8")
)

# Calculate coverage within each store-week
actual_coverage = (
    key_matches.groupby(store_week_keys)
    .agg(
        transaction_products=("PRODUCT_ID", "nunique"),
        matched_products=("MATCHED", "sum")
    )
    .reset_index()
)

actual_coverage["match_percentage"] = (
    100
    * actual_coverage["matched_products"]
    / actual_coverage["transaction_products"]
).round(2)

print("Overall unique transaction keys:", len(key_matches))
print("Overall matching percentage:",
      round(key_matches["MATCHED"].mean() * 100, 2), "%")

print("\nStore-week matching percentage summary:")
print(actual_coverage["match_percentage"].describe())

print("\nLowest matching store-weeks:")
display(actual_coverage.sort_values("match_percentage").head(10))

print("\nHighest matching store-weeks:")
display(
    actual_coverage.sort_values("match_percentage", ascending=False).head(10)
)

Overall unique transaction keys: 2265861
Overall matching percentage: 21.33 %

Store-week matching percentage summary:
count    10045.000000
mean        21.909064
std          6.378682
min          0.000000
25%         17.780000
50%         21.350000
75%         25.410000
max        100.000000
Name: match_percentage, dtype: float64

Lowest matching store-weeks:


,STORE_ID,WEEK_NO,transaction_products,matched_products,match_percentage
1,286,10,20,0,0.0
10029,34280,86,6,0,0.0
13,286,22,16,0,0.0
3530,343,9,1,0,0.0
4926,366,44,2,0,0.0
4933,366,51,12,0,0.0
7365,415,80,2,0,0.0
4980,366,99,9,0,0.0
4953,366,72,4,0,0.0
8507,442,12,9,0,0.0



Highest matching store-weeks:


,STORE_ID,WEEK_NO,transaction_products,matched_products,match_percentage
4951,366,70,1,1,100.00
9968,34007,78,1,1,100.00
9713,31862,10,11,9,81.82
4802,365,13,17,12,70.59
5714,379,94,9,6,66.67
5670,379,47,22,13,59.09
5668,379,45,22,13,59.09
4798,365,9,21,12,57.14
3481,341,53,177,101,57.06
5674,379,51,9,5,55.56


In [41]:
# Step 5: Inspect product IDs for one store-week

store_id = 286
week_no = 10

# Products purchased at this store in this week
transaction_products = set(
    transactions.loc[
        (transactions["STORE_ID"] == store_id)
        & (transactions["WEEK_NO"] == week_no),
        "PRODUCT_ID"
    ].unique()
)

# Products represented in the promotion data for this store-week
promotion_products = set(
    promotions.loc[
        (promotions["STORE_ID"] == store_id)
        & (promotions["WEEK_NO"] == week_no),
        "PRODUCT_ID"
    ].unique()
)

matched_products = transaction_products & promotion_products
unmatched_products = transaction_products - promotion_products

print("Store:", store_id, "| Week:", week_no)
print("Products purchased:", len(transaction_products))
print("Products in promotion data:", len(promotion_products))
print("Matching products:", len(matched_products))
print("Unmatched purchased products:", len(unmatched_products))

print("\nSample purchased product IDs:")
print(sorted(transaction_products)[:15])

print("\nSample promotion product IDs:")
print(sorted(promotion_products)[:15])

print("\nSample unmatched purchased product IDs:")
print(sorted(unmatched_products)[:15])

Store: 286 | Week: 10
Products purchased: 20
Products in promotion data: 1820
Matching products: 0
Unmatched purchased products: 20

Sample purchased product IDs:
[np.int64(854852), np.int64(857849), np.int64(867188), np.int64(874643), np.int64(883370), np.int64(891426), np.int64(972312), np.int64(974904), np.int64(990694), np.int64(995965), np.int64(1029743), np.int64(1032043), np.int64(1057260), np.int64(1082185), np.int64(1089392)]

Sample promotion product IDs:
[np.int64(27951), np.int64(28603), np.int64(28929), np.int64(29132), np.int64(29431), np.int64(30374), np.int64(30808), np.int64(30937), np.int64(31126), np.int64(31170), np.int64(32124), np.int64(32439), np.int64(32456), np.int64(32970), np.int64(33177)]

Sample unmatched purchased product IDs:
[np.int64(854852), np.int64(857849), np.int64(867188), np.int64(874643), np.int64(883370), np.int64(891426), np.int64(972312), np.int64(974904), np.int64(990694), np.int64(995965), np.int64(1029743), np.int64(1032043), np.int64(10572

In [42]:
# Step 6: Trace unmatched products across the promotion dataset

# Reuse the unmatched products from Store 286, Week 10
sample_unmatched = sorted(unmatched_products)

# Check whether each product appears anywhere in promotions
products_found_anywhere = set(
    promotions.loc[
        promotions["PRODUCT_ID"].isin(sample_unmatched),
        "PRODUCT_ID"
    ].unique()
)

# Check whether each product appears at Store 286, in any week
products_at_same_store = set(
    promotions.loc[
        (promotions["STORE_ID"] == store_id)
        & promotions["PRODUCT_ID"].isin(sample_unmatched),
        "PRODUCT_ID"
    ].unique()
)

# Check whether each product appears in Week 10, at any store
products_in_same_week = set(
    promotions.loc[
        (promotions["WEEK_NO"] == week_no)
        & promotions["PRODUCT_ID"].isin(sample_unmatched),
        "PRODUCT_ID"
    ].unique()
)

print("Unmatched products examined:", len(sample_unmatched))

print(
    "Found anywhere in promotions:",
    len(products_found_anywhere)
)

print(
    "Found at Store 286 in any week:",
    len(products_at_same_store)
)

print(
    "Found in Week 10 at any store:",
    len(products_in_same_week)
)

print("\nProduct-level results:")

for product_id in sample_unmatched:
    print(
        f"Product {product_id}: "
        f"anywhere={product_id in products_found_anywhere}, "
        f"same_store={product_id in products_at_same_store}, "
        f"same_week={product_id in products_in_same_week}"
    )

Unmatched products examined: 20
Found anywhere in promotions: 20
Found at Store 286 in any week: 14
Found in Week 10 at any store: 7

Product-level results:
Product 854852: anywhere=True, same_store=True, same_week=False
Product 857849: anywhere=True, same_store=True, same_week=True
Product 867188: anywhere=True, same_store=True, same_week=True
Product 874643: anywhere=True, same_store=True, same_week=True
Product 883370: anywhere=True, same_store=True, same_week=False
Product 891426: anywhere=True, same_store=True, same_week=False
Product 972312: anywhere=True, same_store=True, same_week=False
Product 974904: anywhere=True, same_store=False, same_week=False
Product 990694: anywhere=True, same_store=True, same_week=False
Product 995965: anywhere=True, same_store=True, same_week=False
Product 1029743: anywhere=True, same_store=True, same_week=True
Product 1032043: anywhere=True, same_store=False, same_week=False
Product 1057260: anywhere=True, same_store=False, same_week=False
Product 1

In [43]:
# Step 1: Aggregate transactions by product, store, and week

import numpy as np
import pandas as pd

analysis_keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

transaction_weekly = (
    transactions
    .groupby(analysis_keys, as_index=False)
    .agg(
        TOTAL_QUANTITY=("QUANTITY", "sum"),
        TOTAL_SALES=("SALES_VALUE", "sum"),
        TOTAL_RETAIL_DISC=("RETAIL_DISC", "sum"),
        TOTAL_COUPON_DISC=("COUPON_DISC", "sum"),
        TOTAL_COUPON_MATCH_DISC=("COUPON_MATCH_DISC", "sum"),
        TRANSACTION_ROWS=("BASKET_ID", "size"),
        UNIQUE_BASKETS=("BASKET_ID", "nunique")
    )
)

# Preliminary weighted unit-price proxy
transaction_weekly["PRICE_PROXY"] = np.where(
    transaction_weekly["TOTAL_QUANTITY"] > 0,
    transaction_weekly["TOTAL_SALES"]
    / transaction_weekly["TOTAL_QUANTITY"],
    np.nan
)

print("Original transaction rows:", len(transactions))
print("Aggregated product-store-week rows:", len(transaction_weekly))
print("Duplicate analysis keys:",
      transaction_weekly.duplicated(subset=analysis_keys).sum())

print("\nSummary of aggregated data:")
display(transaction_weekly.head())

print("\nQuantity and price proxy summary:")
display(
    transaction_weekly[
        ["TOTAL_QUANTITY", "TOTAL_SALES", "PRICE_PROXY"]
    ].describe()
)

print("\nRecords with non-positive total quantity:",
      (transaction_weekly["TOTAL_QUANTITY"] <= 0).sum())

print("Records with missing price proxy:",
      transaction_weekly["PRICE_PROXY"].isna().sum())

Original transaction rows: 2595732
Aggregated product-store-week rows: 2370784
Duplicate analysis keys: 0

Summary of aggregated data:


,PRODUCT_ID,STORE_ID,WEEK_NO,TOTAL_QUANTITY,TOTAL_SALES,TOTAL_RETAIL_DISC,TOTAL_COUPON_DISC,TOTAL_COUPON_MATCH_DISC,TRANSACTION_ROWS,UNIQUE_BASKETS,PRICE_PROXY
0,25671,3191,59,4,13.96,0.0,0.0,0.0,1,1,3.49
1,25671,3266,36,1,3.49,0.0,0.0,0.0,1,1,3.49
2,25671,3313,23,1,3.49,0.0,0.0,0.0,1,1,3.49
3,26081,3235,36,1,0.99,0.0,0.0,0.0,1,1,0.99
4,26093,33904,66,1,1.59,0.0,0.0,0.0,1,1,1.59



Quantity and price proxy summary:


,TOTAL_QUANTITY,TOTAL_SALES,PRICE_PROXY
count,2.370784e+06,2.370784e+06,2.359168e+06
mean,1.099576e+02,3.398649e+00,2.497246e+00
std,3.025345e+03,8.002661e+00,2.761353e+00
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.390000e+00,1.000000e+00
50%,1.000000e+00,2.290000e+00,1.990000e+00
75%,1.000000e+00,3.760000e+00,2.990000e+00
max,2.623010e+05,8.400000e+02,4.999900e+02



Records with non-positive total quantity: 11616
Records with missing price proxy: 11616


In [44]:

import pandas as pd
import numpy as np
import json
from pathlib import Path

# --------------------------------------------------
# 1. Define output directory
# --------------------------------------------------
output_dir = Path("/kaggle/working/causal_variable_selection")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Aggregate transactions to product-store-week
# --------------------------------------------------
keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

transaction_weekly_final = (
    transactions.groupby(keys, as_index=False)
    .agg(
        TOTAL_QUANTITY=("QUANTITY", "sum"),
        TOTAL_SALES=("SALES_VALUE", "sum"),
        TOTAL_RETAIL_DISC=("RETAIL_DISC", "sum"),
        TOTAL_COUPON_DISC=("COUPON_DISC", "sum"),
        TOTAL_COUPON_MATCH_DISC=("COUPON_MATCH_DISC", "sum"),
        TRANSACTION_ROWS=("BASKET_ID", "size"),
        UNIQUE_BASKETS=("BASKET_ID", "nunique"),
        ZERO_QUANTITY_ROWS=("ZERO_QUANTITY_FLAG", "sum"),
        ZERO_SALES_ROWS=("ZERO_SALES_FLAG", "sum"),
        EXTREME_QUANTITY_ROWS=("EXTREME_QUANTITY_FLAG", "sum")
    )
)

# --------------------------------------------------
# 3. Engineer preliminary price and data-quality flags
# --------------------------------------------------
valid_quantity = transaction_weekly_final["TOTAL_QUANTITY"] > 0

transaction_weekly_final["PRICE_PROXY"] = np.nan
transaction_weekly_final.loc[valid_quantity, "PRICE_PROXY"] = (
    transaction_weekly_final.loc[valid_quantity, "TOTAL_SALES"]
    / transaction_weekly_final.loc[valid_quantity, "TOTAL_QUANTITY"]
)

transaction_weekly_final["ZERO_OR_NEGATIVE_QUANTITY_FLAG"] = (
    transaction_weekly_final["TOTAL_QUANTITY"] <= 0
).astype("int8")

transaction_weekly_final["ZERO_SALES_FLAG"] = (
    transaction_weekly_final["TOTAL_SALES"] == 0
).astype("int8")

transaction_weekly_final["MISSING_PRICE_PROXY_FLAG"] = (
    transaction_weekly_final["PRICE_PROXY"].isna()
).astype("int8")

transaction_weekly_final["EXTREME_QUANTITY_FLAG"] = (
    transaction_weekly_final["TOTAL_QUANTITY"] > 1000
).astype("int8")

# Retain negative or zero price values for review; do not silently delete.
transaction_weekly_final["NONPOSITIVE_PRICE_PROXY_FLAG"] = (
    transaction_weekly_final["PRICE_PROXY"].notna()
    & (transaction_weekly_final["PRICE_PROXY"] <= 0)
).astype("int8")

# --------------------------------------------------
# 4. Create a feature dictionary
# --------------------------------------------------
feature_dictionary = pd.DataFrame([
    ("PRODUCT_ID", "Product identifier", "Identifier"),
    ("STORE_ID", "Store identifier", "Identifier"),
    ("WEEK_NO", "Week number", "Time index"),
    ("TOTAL_QUANTITY", "Sum of purchased quantity in the product-store-week", "Outcome candidate"),
    ("TOTAL_SALES", "Sum of recorded sales value", "Sales measure"),
    ("TOTAL_RETAIL_DISC", "Sum of retail discount values", "Candidate control; semantics to verify"),
    ("TOTAL_COUPON_DISC", "Sum of coupon discount values", "Candidate control; semantics to verify"),
    ("TOTAL_COUPON_MATCH_DISC", "Sum of coupon match discount values", "Candidate control; semantics to verify"),
    ("TRANSACTION_ROWS", "Number of source transaction rows aggregated", "Engineered feature"),
    ("UNIQUE_BASKETS", "Number of distinct baskets represented", "Engineered feature"),
    ("ZERO_QUANTITY_ROWS", "Number of source rows with zero quantity", "Data-quality feature"),
    ("ZERO_SALES_ROWS", "Number of source rows with zero sales", "Data-quality feature"),
    ("EXTREME_QUANTITY_ROWS", "Number of source rows with quantity greater than 1000", "Data-quality feature"),
    ("PRICE_PROXY", "TOTAL_SALES divided by TOTAL_QUANTITY when quantity is positive", "Preliminary treatment candidate; not verified as shelf price"),
    ("ZERO_OR_NEGATIVE_QUANTITY_FLAG", "1 when aggregated quantity is non-positive", "Data-quality flag"),
    ("ZERO_SALES_FLAG", "1 when aggregated sales value is zero", "Data-quality flag"),
    ("MISSING_PRICE_PROXY_FLAG", "1 when the preliminary price proxy is missing", "Data-quality flag"),
    ("EXTREME_QUANTITY_FLAG", "1 when aggregated quantity exceeds 1000", "Data-quality flag"),
    ("NONPOSITIVE_PRICE_PROXY_FLAG", "1 when a non-missing price proxy is non-positive", "Data-quality flag")
], columns=["COLUMN_NAME", "DESCRIPTION", "ROLE"])

# --------------------------------------------------
# 5. Create a concise processing report
# --------------------------------------------------
report = {
    "status": "Feature engineering checkpoint",
    "aggregation_level": "One row per PRODUCT_ID, STORE_ID, WEEK_NO",
    "source_transaction_rows": int(len(transactions)),
    "engineered_rows": int(len(transaction_weekly_final)),
    "duplicate_key_count": int(
        transaction_weekly_final.duplicated(subset=keys).sum()
    ),
    "missing_price_proxy_count": int(
        transaction_weekly_final["PRICE_PROXY"].isna().sum()
    ),
    "nonpositive_quantity_count": int(
        (transaction_weekly_final["TOTAL_QUANTITY"] <= 0).sum()
    ),
    "nonpositive_price_proxy_count": int(
        transaction_weekly_final["NONPOSITIVE_PRICE_PROXY_FLAG"].sum()
    ),
    "promotion_integration": (
        "Not merged because only 21.33% of eligible unique transaction "
        "keys matched promotion keys. Missing promotion records are not "
        "assumed to mean no promotion."
    ),
    "price_definition": (
        "PRICE_PROXY is a preliminary ratio, not a verified shelf price."
    ),
    "dml_training": "Out of scope"
}

# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------
transaction_weekly_final.to_csv(
    output_dir / "transaction_weekly_final.csv",
    index=False
)

feature_dictionary.to_csv(
    output_dir / "feature_dictionary.csv",
    index=False
)

with open(output_dir / "feature_engineering_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Feature engineering completed.")
print("Output directory:", output_dir)
print("Final rows:", len(transaction_weekly_final))
print("Duplicate keys:", transaction_weekly_final.duplicated(subset=keys).sum())
print("Saved files:")
for file in [
    "transaction_weekly_final.csv",
    "feature_dictionary.csv",
    "feature_engineering_report.json"
]:
    path = output_dir / file
    print(f"- {file}: {path.stat().st_size / (1024**2):.2f} MB")


Feature engineering completed.
Output directory: /kaggle/working/causal_variable_selection
Final rows: 2370784
Duplicate keys: 0
Saved files:
- transaction_weekly_final.csv: 134.95 MB
- feature_dictionary.csv: 0.00 MB
- feature_engineering_report.json: 0.00 MB
